In [ ]:
import os
import pandas as pd

# Cap Polars to 6 threads
os.environ["POLARS_MAX_THREADS"] = "6"

import polars as pl

# loading the data set, using 2024 data
df_lazy = pl.scan_parquet("aisdk-parquet/aisdk-2024-1h.parquet")

# Quick sanity check: pull just the first 100k rows into memory
# to inspect schema and spot any obvious data quality issues
df_sample = df_lazy.head(100_000).collect()

# checking ship types
df_lazy.group_by("Ship type").len().sort("len", descending=True).collect()

In [ ]:
import plotly.express as px
import plotly.io as pio
import pandas as pd

pio.renderers.default = "vscode"

# Pull only the two emergency device types from the full lazy dataset.
emergency = (
    df_lazy
    .filter(pl.col("Type of mobile").is_in(["Search and Rescue Transponder", "Man Overboard Device"]))
    .filter(
        # Coordinate guards remove the common (0, 0) AIS artifact and any out-of-range values.
        pl.col("Latitude").is_between(-90, 90) &
        pl.col("Longitude").is_between(-180, 180) &
        (pl.col("Latitude") != 0) &
        (pl.col("Longitude") != 0)
    ) # Sorting by timestamp here means the cumulative frame logic later stays correct
    .select(["MMSI", "# Timestamp", "Latitude", "Longitude", "Name", "Type of mobile"])
    .sort("# Timestamp")
    .collect()
    .to_pandas()
)

# Collapse per-minute pings into daily buckets.
# This reduces ~300k rows to ~2k and makes the animation readable
emergency["Time"] = emergency["# Timestamp"].dt.strftime("%Y-%m-%d")

# Assign a render order so Man Overboard dots (rarer, more critical)
# draw on top of SAR dots when they share the same coordinates.
emergency["sort_order"] = emergency["Type of mobile"].map({
    "Search and Rescue Transponder": 0,
    "Man Overboard Device": 1
})

# Sort before deduplicating so that when two rows would otherwise collapse
# into one, the MOB row (higher sort_order) is the one that survives.
emergency = emergency.sort_values("sort_order")
emergency = emergency.drop_duplicates(subset=["MMSI", "Time", "Type of mobile"])


first_day = sorted(emergency["Time"].unique())[0]
for mobile_type in ["Search and Rescue Transponder", "Man Overboard Device"]:
    if not ((emergency["Time"] == first_day) & (emergency["Type of mobile"] == mobile_type)).any():
        dummy = pd.DataFrame([{
            "MMSI": -1,
            "Latitude": 0.001,
            "Longitude": 0.001,
            "Name": "",
            "Type of mobile": mobile_type,
            "Time": first_day,
            "sort_order": 0,
            "# Timestamp": pd.Timestamp(first_day)
        }])
        emergency = pd.concat([emergency, dummy], ignore_index=True)

print(f"Unique days: {emergency['Time'].nunique()}")
print(f"Total points: {len(emergency)}")

# Build cumulative frames: each frame contains all pings up to that date,
# not just that day's pings. This creates a "history accumulates" effect
# where dots persist on the map rather than blinking in and out.
frames = []
for time in sorted(emergency["Time"].unique()):
    frame = emergency[emergency["Time"] <= time].copy()
    frame["frame"] = time
    frames.append(frame)

# This produces a large dataframe: ~2k points × 337 days ≈ 326k rows.
# Polars or chunked concat would be faster for larger datasets.
animation_df = pd.concat(frames)
print(f"Animation dataframe size: {len(animation_df)} rows")

fig = px.scatter_map(
    animation_df,
    lat="Latitude",
    lon="Longitude",
    color="Type of mobile",
    color_discrete_map={
        "Search and Rescue Transponder": "red",
        "Man Overboard Device": "orange",
    },
    animation_frame="frame",   # One slider step per day
    hover_data={"MMSI": True, "Name": True, "Time": True},
    title="Emergency Alerts Over Time",
    zoom=3,
)

fig.update_traces(marker=dict(size=10))

fig.update_layout(
    map_style="carto-positron",
    margin={"r": 0, "t": 40, "l": 0, "b": 0},
    height=600,
    # Override Plotly's default animation controls with custom Play/Pause buttons.
    updatemenus=[{
        "type": "buttons",
        "showactive": False,
        "buttons": [
            {"label": "▶ Play",  "method": "animate", "args": [None, {"frame": {"duration": 100}, "fromcurrent": True}]},
            {"label": "⏸ Pause", "method": "animate", "args": [[None], {"mode": "immediate"}]}
        ]
    }]
)

fig.show()